In [1]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI 
from dotenv import load_dotenv
from IPython.display import Markdown 
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import wrap_model_call
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_openai.middleware import OpenAIModerationMiddleware
from langchain.agents.middleware import (
    wrap_model_call,
    wrap_tool_call,
    dynamic_prompt,
    HumanInTheLoopMiddleware,
    ModelRequest,
    ModelResponse,
)
from langchain.agents.middleware import after_model, hook_config, AgentState
from langgraph.runtime import Runtime


c:\Users\DELL\Documents\agents\agentlangchain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
load_dotenv()

True

In [3]:
basic_llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)
advanced_llm=ChatOpenAI(model="gpt-4o", temperature=0)

In [4]:
agent=create_agent(
    model=basic_llm,
    tools=[],
    system_prompt="You are a helpful assistant"
)

In [14]:
resp=agent.invoke(input={
    "messages":[
        {"role": "user", "content": "Je m'appelle John"}
    ]
})
print(resp['messages'][-1].content)

Bonjour, John ! Comment puis-je vous aider aujourd'hui ?


In [ ]:
@wrap_model_call
def dynamic_model_selection(request, handler):
    context = request.runtime.context or {}   

    env = context.get("env", "test")

    if env == "prod":
        print("Using advanced model")
        return handler(request.override(model=advanced_llm))
    else:
        print("Using basic model")
        return handler(request.override(model=basic_llm))

In [16]:
agent2 = create_agent(
    model=basic_llm,
    middleware=[dynamic_model_selection],
    tools=[],
    debug=True
)

In [17]:
response = agent2.invoke(
    {"messages": [{"role": "user", "content": "cest quoi les avantages de l'ia"}]},
    context={"env": "test"}  
)

[values] {'messages': [HumanMessage(content="cest quoi les avantages de l'ia", additional_kwargs={}, response_metadata={}, id='8ab3747e-8502-4ae8-8546-841654b2c6fc')]}
Using basic model
[updates] {'model': {'messages': [AIMessage(content="L'intelligence artificielle (IA) offre de nombreux avantages dans divers domaines. Voici quelques-uns des principaux avantages :\n\n1. **Automatisation des tâches** : L'IA peut automatiser des tâches répétitives et chronophages, permettant aux employés de se concentrer sur des activités à plus forte valeur ajoutée.\n\n2. **Analyse de données** : L'IA peut traiter et analyser de grandes quantités de données rapidement et avec précision, ce qui aide à prendre des décisions éclairées basées sur des données.\n\n3. **Personnalisation** : Dans le domaine du marketing et du service client, l'IA permet de personnaliser les expériences utilisateur en analysant les comportements et les préférences des clients.\n\n4. **Amélioration de l'efficacité** : Les systèm

In [18]:
print(response['messages'][-1].content)

L'intelligence artificielle (IA) offre de nombreux avantages dans divers domaines. Voici quelques-uns des principaux avantages :

1. **Automatisation des tâches** : L'IA peut automatiser des tâches répétitives et chronophages, permettant aux employés de se concentrer sur des activités à plus forte valeur ajoutée.

2. **Analyse de données** : L'IA peut traiter et analyser de grandes quantités de données rapidement et avec précision, ce qui aide à prendre des décisions éclairées basées sur des données.

3. **Personnalisation** : Dans le domaine du marketing et du service client, l'IA permet de personnaliser les expériences utilisateur en analysant les comportements et les préférences des clients.

4. **Amélioration de l'efficacité** : Les systèmes d'IA peuvent optimiser les processus opérationnels, réduire les coûts et améliorer la productivité.

5. **Précision et fiabilité** : Les algorithmes d'IA peuvent réduire les erreurs humaines et fournir des résultats plus précis dans des domaine

In [19]:
print(display(Markdown(response['messages'][-1].content)))

L'intelligence artificielle (IA) offre de nombreux avantages dans divers domaines. Voici quelques-uns des principaux avantages :

1. **Automatisation des tâches** : L'IA peut automatiser des tâches répétitives et chronophages, permettant aux employés de se concentrer sur des activités à plus forte valeur ajoutée.

2. **Analyse de données** : L'IA peut traiter et analyser de grandes quantités de données rapidement et avec précision, ce qui aide à prendre des décisions éclairées basées sur des données.

3. **Personnalisation** : Dans le domaine du marketing et du service client, l'IA permet de personnaliser les expériences utilisateur en analysant les comportements et les préférences des clients.

4. **Amélioration de l'efficacité** : Les systèmes d'IA peuvent optimiser les processus opérationnels, réduire les coûts et améliorer la productivité.

5. **Précision et fiabilité** : Les algorithmes d'IA peuvent réduire les erreurs humaines et fournir des résultats plus précis dans des domaines comme la médecine, la finance et l'ingénierie.

6. **Accessibilité** : L'IA peut aider à rendre des services et des informations plus accessibles, par exemple, grâce à des assistants virtuels ou des technologies d'assistance pour les personnes handicapées.

7. **Innovation** : L'IA stimule l'innovation en permettant le développement de nouveaux produits et services, ainsi que l'amélioration de ceux qui existent déjà.

8. **Prédiction et anticipation** : Les modèles prédictifs basés sur l'IA peuvent anticiper des tendances et des comportements, ce qui est particulièrement utile dans des domaines comme la finance, la santé et la logistique.

9. **Support à la prise de décision** : L'IA peut fournir des recommandations basées sur des analyses de données, aidant ainsi les décideurs à faire des choix plus éclairés.

10. **Amélioration de la sécurité** : Dans des domaines comme la cybersécurité, l'IA peut détecter des anomalies et des menaces en temps réel, renforçant ainsi la sécurité des systèmes.

Ces avantages montrent comment l'IA peut transformer divers secteurs et améliorer la qualité de vie, tout en soulevant également des questions éthiques et des défis à relever.

None


In [20]:
messages = [
    HumanMessage(content="Bonjour"),
    HumanMessage(content="Je m'appelle Aya"),
    HumanMessage(content="Quel est mon nom ?")
]

In [104]:
agent3 = create_agent(
    model=basic_llm,
    middleware=[dynamic_model_selection],
    tools=[],
)

In [105]:
resp = agent3.invoke({
    "messages": messages
})

print(resp["messages"][-1].content)

Using basic model
Votre nom est Aya. En quoi puis-je vous aider aujourd'hui ?


# agent avec mémoire


In [ ]:
memory = InMemorySaver()

In [35]:
agent4 = create_agent(
    model=basic_llm,
    middleware=[dynamic_model_selection],
    tools=[],
    checkpointer=memory,
    debug=True
)

In [39]:
# message 1
agent4.invoke(
    {"messages": [{"role": "user", "content": "Je m'appelle Aya"}]},
    config={"configurable": {"thread_id": "session1"}}
)


[values] {'messages': [HumanMessage(content="Je m'appelle Aya", additional_kwargs={}, response_metadata={}, id='025c6f7b-c1aa-400b-896e-a0e7611f57b8')]}
Using basic model
[updates] {'model': {'messages': [AIMessage(content="Enchanté, Aya ! Comment puis-je vous aider aujourd'hui ?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 11, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO1QhpFcG5s49ZlubriYcOCECuf94', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f75-91ed-7c03-8e94-fe371b5bdb33-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 13,

{'messages': [HumanMessage(content="Je m'appelle Aya", additional_kwargs={}, response_metadata={}, id='025c6f7b-c1aa-400b-896e-a0e7611f57b8'),
  AIMessage(content="Enchanté, Aya ! Comment puis-je vous aider aujourd'hui ?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 11, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO1QhpFcG5s49ZlubriYcOCECuf94', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f75-91ed-7c03-8e94-fe371b5bdb33-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 13, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'ca

In [41]:
# message 2
resp = agent4.invoke(
    {"messages": [{"role": "user", "content": "Quel est mon nom ?"}]},
    config={"configurable": {"thread_id": "session1"}}

)

print(resp["messages"][-1].content)

[values] {'messages': [HumanMessage(content="Je m'appelle Aya", additional_kwargs={}, response_metadata={}, id='025c6f7b-c1aa-400b-896e-a0e7611f57b8'), AIMessage(content="Enchanté, Aya ! Comment puis-je vous aider aujourd'hui ?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 11, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO1QhpFcG5s49ZlubriYcOCECuf94', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f75-91ed-7c03-8e94-fe371b5bdb33-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 13, 'total_tokens': 24, 'input_token_details': {'audio':

In [ ]:
@tool
def get_current_weather(location: str) -> dict:
    """Get the current weather in a given location"""
    
    print(f"Getting weather for {location}")
    
    return { 
        "city": location,
        "temperature": "25°C", 
        "humidity": "60%",
        "pressure": "1013 hPa"
    }

In [53]:
@tool
def get_employee_info(employee_name: str) -> str:
    """
    Get information about the given employee.
    """
    employees = {
        "aya": "Aya est étudiante en génie informatique.",
        "mohamed": "Mohamed travaille dans l'équipe data.",
        "john": "John est développeur Python."
    }

    name = employee_name.lower()

    if name in employees:
        return employees[name]
    return f"Aucune information trouvée pour {employee_name}."

In [58]:
agent5 = create_agent(
    model=basic_llm,
    tools=[get_current_weather, get_employee_info],
    checkpointer=memory,
    system_prompt="answer the user qst using the tools if needed"
)

In [60]:
config={"configurable": {"thread_id": "session1"}}
resp = agent5.invoke(
    input={"messages": [{"role": "user", "content": "Quel est le temps à Paris ?"}]},config=config)
print(resp["messages"][-1].content)

Getting weather for Paris
Le temps à Paris est actuellement de 25°C avec une humidité de 60% et une pression de 1013 hPa. Si vous avez besoin d'autres informations, n'hésitez pas à demander !


In [61]:
config={"configurable": {"thread_id": "session1"}}
resp = agent5.invoke(
    input={"messages": [{"role": "user", "content": "Quel est mon nom"}]},config=config)
print(resp["messages"][-1].content)

Votre nom est Aya. Si vous avez d'autres questions ou si vous souhaitez discuter de quelque chose en particulier, n'hésitez pas à me le faire savoir !


In [68]:
duck_tool = DuckDuckGoSearchRun()
tavily_tool = TavilySearchResults(max_results=3)
python_tool = PythonREPLTool()

C:\Users\DELL\AppData\Local\Temp\ipykernel_27520\267950013.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=3)


In [69]:
tools = [duck_tool, tavily_tool, python_tool]

for tool in tools:
    print(tool.name)

duckduckgo_search
tavily_search_results_json
Python_REPL


In [70]:
agent6 = create_agent(
    model=basic_llm,
    tools=tools,
    checkpointer=memory,
)


In [73]:
resp = agent6.invoke({
    "messages": [
        {"role": "user", "content": "Quel est le temps à Paris ?"}
    ]
},
config={"configurable": {"thread_id": "session1"}}
)
    
print(resp["messages"][-1].content)

Le temps à Paris aujourd'hui est sec avec quelques nuages. Voici les températures prévues :

- Matin : 6°C
- Après-midi : 16°C
- Soir : 12°C
- Nuit : 9°C

Si vous avez besoin d'autres informations ou d'une mise à jour, n'hésitez pas à demander !


In [75]:
resp = agent6.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Utilise Tavily pour chercher les applications de l'intelligence artificielle en médecine."
        }
    ]
},
config={"configurable": {"thread_id": "session1"}})

print(resp["messages"][-1].content)

L'intelligence artificielle (IA) a de nombreuses applications en médecine, et elle transforme le secteur de manière significative. Voici quelques-unes des principales applications :

1. **Amélioration des diagnostics** : L'IA aide à améliorer la précision des diagnostics en analysant des données médicales et en identifiant des modèles que les médecins pourraient manquer.

2. **Traitement d'images médicales** : Les domaines comme la radiologie, la dermatologie et l'ophtalmologie bénéficient particulièrement de l'IA pour l'analyse d'images, permettant une détection précoce de maladies.

3. **Automatisation des tâches administratives** : L'IA peut automatiser des tâches répétitives, permettant aux professionnels de santé de se concentrer davantage sur les soins aux patients.

4. **Optimisation des traitements** : L'IA permet de personnaliser les traitements en tenant compte des caractéristiques biologiques et sociales des patients, ce qui peut améliorer les résultats.

5. **Prise de décis

In [79]:
python_tool.invoke("print(sum([i*i for i in range(1,11)]))")


'385\n'

In [82]:
resp = agent6.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Utilise Python pour calculer la somme des carrés de 1 à 10, puis affiche clairement le résultat final."
        }
    ]
},
config={"configurable": {"thread_id": "session1"}})

In [83]:

print(resp["messages"][-1].content)

La somme des carrés de 1 à 10 est **385**.


In [ ]:
resp = agent6.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Cherche avec DuckDuckGo ce qu'est le machine learning, puis utilise Python pour donner un petit exemple numérique."
        }
    ]
},
config={"configurable": {"thread_id": "session1"}})

print(resp["messages"][-1].content)

# test combiné

In [87]:
resp = agent6.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Cherche avec DuckDuckGo ce qu'est le machine learning, puis utilise Python pour donner un petit exemple numérique."
        }
    ]
},
config={"configurable": {"thread_id": "session1"}})

print(resp["messages"][-1].content)

### Qu'est-ce que le Machine Learning ?

Le machine learning, ou apprentissage automatique en français, est un sous-domaine de l'intelligence artificielle (IA) qui permet aux machines d'apprendre à partir de données. Au lieu d'être explicitement programmées pour effectuer une tâche, les machines utilisent des algorithmes pour identifier des modèles dans les données et faire des prédictions ou des décisions basées sur ces modèles. Le deep learning est un sous-ensemble du machine learning qui utilise des réseaux de neurones profonds pour traiter des données complexes.

### Exemple Numérique

Malheureusement, il semble que le module `sklearn` (scikit-learn) n'est pas disponible dans l'environnement actuel, ce qui empêche l'exécution d'un exemple de machine learning. Cependant, je peux vous expliquer comment un exemple simple pourrait fonctionner :

1. **Données** : Supposons que nous avons des données sur des caractéristiques (X) et des cibles (y).
   - Caractéristiques : [1, 2, 3, 4, 5]


Nous avons enrichi l’agent en lui ajoutant des tools prédéfinis. DuckDuckGoSearchRun permet d’effectuer des recherches web rapides, TavilySearchResults fournit une recherche web adaptée aux agents intelligents, et PythonREPLTool permet l’exécution de code Python pour réaliser des calculs et des traitements logiques. Les tools sont transmis à create_agent, qui laisse ensuite le modèle décider du moment opportun pour les appeler

### Aouter des middlewares à l'agent : dynamic_model, dynamic_prompt, tool_error_handling, GardRails, Human In The Loop

In [89]:
@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location."""
    return f"La météo à {location} est ensoleillée, 25°C."

@tool
def risky_tool(text: str) -> str:
    """Tool sensible pour démontrer Human In The Loop."""
    return f"Action sensible exécutée avec : {text}"

In [91]:
@wrap_tool_call
def tool_error_handling(request, handler):
    try:
        print(f"Tool appelé : {request.tool_call['name']}")
        return handler(request)
    except Exception as e:
        print("Erreur tool :", e)
        return ToolMessage(
            content=f"Erreur lors de l'exécution du tool : {str(e)}",
            tool_call_id=request.tool_call["id"]
        )

In [95]:
@after_model
@hook_config(can_jump_to=["end"])
def simple_guardrails(state: AgentState, runtime):
    last_message = state["messages"][-1]

    blocked_words = ["pirater", "malware", "voler mot de passe"]

    content = str(last_message.content).lower()
    if any(word in content for word in blocked_words):
        return {
            "messages": [AIMessage(content="Je ne peux pas aider pour cette demande.")],
            "jump_to": "end"
        }

    return None

### Human In The Loop


In [96]:
memory = InMemorySaver()

hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "risky_tool": {
            "allowed_decisions": ["approve", "edit", "reject"]
        },
        "get_current_weather": False
    },
    description_prefix="Validation humaine requise pour le tool"
)

In [99]:
agent_middleware = create_agent(
    model=basic_llm,
    tools=[get_current_weather, risky_tool],
    middleware=[
        dynamic_model_selection,
        tool_error_handling,
        simple_guardrails,         
        hitl_middleware
    ],
    checkpointer=memory,
)

In [100]:
resp = agent_middleware.invoke(
    {
        "messages": [
            {"role": "user", "content": "Explique le machine learning"}
        ]
    },
    context={"env": "test", "user_level": "beginner"},
    config={"configurable": {"thread_id": "session1"}}
)

print(resp["messages"][-1].content)

Using basic model
Le machine learning, ou apprentissage automatique, est une branche de l'intelligence artificielle qui permet aux ordinateurs d'apprendre à partir de données et d'améliorer leurs performances sur des tâches spécifiques sans être explicitement programmés pour chaque tâche. Voici une explication détaillée des concepts clés du machine learning :

### 1. **Données**
Le machine learning repose sur des ensembles de données. Ces données peuvent être de différents types, comme des images, des textes, des chiffres, etc. Elles sont essentielles pour entraîner les modèles.

### 2. **Modèles**
Un modèle est une représentation mathématique qui apprend à partir des données. Il peut être considéré comme une fonction qui prend des entrées (données) et produit des sorties (prédictions). Les modèles peuvent varier en complexité, allant de simples régressions linéaires à des réseaux de neurones profonds.

### 3. **Entraînement**
Le processus d'entraînement consiste à ajuster les paramètr

In [101]:
resp = agent_middleware.invoke(
    {
        "messages": [
            {"role": "user", "content": "Explique en détail les différences entre RAG, fine-tuning et agents."}
        ]
    },
    context={"env": "prod", "user_level": "expert"},
    config={"configurable": {"thread_id": "session1"}}
)

print(resp["messages"][-1].content)

Using advanced model
Les concepts de RAG (Retrieval-Augmented Generation), fine-tuning et agents sont tous liés à l'amélioration des capacités des modèles d'intelligence artificielle, mais ils diffèrent dans leur approche et leur application. Voici une explication détaillée de chacun :

### 1. **RAG (Retrieval-Augmented Generation)**
RAG est une technique qui combine la génération de texte avec la récupération d'informations. Elle est particulièrement utile pour les tâches où le modèle doit générer des réponses basées sur des informations spécifiques et à jour.

- **Récupération d'informations** : Avant de générer une réponse, le modèle récupère des informations pertinentes à partir d'une base de données ou d'un ensemble de documents. Cela permet d'enrichir le contexte et d'améliorer la précision des réponses.
  
- **Génération de texte** : Une fois les informations récupérées, le modèle utilise ces données pour générer une réponse cohérente et pertinente.

- **Applications** : RAG est

In [102]:
resp = agent_middleware.invoke(
    {
        "messages": [
            {"role": "user", "content": "Comment pirater un compte ?"}
        ]
    },
    context={"env": "test", "user_level": "beginner"},
    config={"configurable": {"thread_id": "session2"}}
)

print(resp["messages"][-1].content)

Using basic model
Je ne peux pas vous aider avec ça.


In [103]:
resp = agent_middleware.invoke(
    {
        "messages": [
            {"role": "user", "content": "Utilise risky_tool avec le texte: supprimer les données"}
        ]
    },
    context={"env": "prod", "user_level": "expert"},
    config={"configurable": {"thread_id": "session3"}}
)

print(resp)

Using advanced model
{'messages': [HumanMessage(content='Utilise risky_tool avec le texte: supprimer les données', additional_kwargs={}, response_metadata={}, id='c9de2af4-56fe-40ec-b88d-98e4ec58946d'), AIMessage(content="Je ne peux pas exécuter cette action car elle pourrait être risquée ou inappropriée. Si vous avez besoin d'aide pour autre chose, n'hésitez pas à demander.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 82, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_26f5907b3f', 'id': 'chatcmpl-DO2pCy9aOnr2L1wV1Eps6KGztHHTq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2fc7-963b-7ed1-bdbb-d4a748f5d0ff-0', t

Nous avons ajouté plusieurs middlewares à l’agent LangChain. Le middleware dynamic_model permet de choisir dynamiquement le modèle selon le contexte d’exécution. Le middleware dynamic_prompt adapte le prompt système selon le niveau de l’utilisateur. Le middleware tool_error_handling intercepte les erreurs lors des appels tools. Les GuardRails filtrent les contenus non autorisés ou sensibles. Enfin, le Human In The Loop permet d’interrompre l’exécution avant certains tools sensibles afin d’obtenir une validation humaine.